# Check dte/dt using stored energy

In [40]:
# Reload Process each time (keep editable install up-to-date)
%load_ext autoreload
%autoreload 2
from IPython.display import clear_output
from process.main import SingleRun
import process.impurity_radiation as impurity_radiation
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.integrate import solve_ivp
import process.data_structure.physics_variables as pv
import process.data_structure.impurity_radiation_module as irm
import pandas as pd
import process.data_structure
import os
import process

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [41]:
# Set up
input_file = "data/lt_sol_IN.DAT"
single_run = SingleRun(input_file)


# Evaluation derivatives
def eval_process(te, ne, pinj, rfuel, fw, triang):
    single_run = SingleRun(input_file)
    # Loading single run overwrites variables: set again
    process.data_structure.current_drive_variables.p_hcd_primary_extra_heat_mw = pinj
    pv.molflow_plasma_fuelling_required = rfuel
    process.data_structure.impurity_radiation_module.f_nd_impurity_electron_array[
        13
    ] = fw
    pv.triang = triang

    # Solution vector
    pv.temp_plasma_electron_vol_avg_kev = te
    pv.nd_plasma_electrons_vol_avg = ne
    # Clear all output, but show exceptions
    try:
        single_run.run()
        clear_output()
    except:
        clear_output()
        raise

    # raise Exception
    return single_run


The IN.DAT file does not contain any obsolete variables.
 
**************************************************************************************************************
************************************************** PROCESS ***************************************************
************************************** Power Reactor Optimisation Code ***************************************
**************************************************************************************************************
 
Version : 3.2.1
Git Tag : 
Git Branch : 
Date : 26/02/2026 UTC
Time : 10:33
User : jon
Computer : jon-Precision-3560
Directory : /home/jon/code/notebooks/solving_process/solutions_to_process_4
Input : /home/jon/code/notebooks/solving_process/solutions_to_process_4/data/lt_sol_IN.DAT
Run title : generic large tokamak
Run type : Reactor concept design: Pulsed tokamak model, (c) UK Atomic Energy Authority
 
***************************************************************************************

In [42]:
def derivatives(t, y, pinj, rfuel, fw, triang):
    # Scale up to real values
    y_real = y
    te = y_real[0]
    ne = y_real[1]

    # Evaluate PPB and Fuel Equilibrium
    single_run = eval_process(te, ne, pinj, rfuel, fw, triang)
    ppb = process.constraints.constraint_equation_2().constraint_error
    fe = process.constraints.constraint_equation_93().constraint_error
    ni = pv.nd_plasma_ions_total_vol_avg
    vol = pv.vol_plasma
    # Convert to keV
    dte_dt = (
        (2 / 3) * (1 / 1.602e-19) * ((ppb * 1e6 * vol) / ((ni + ne) * vol))
    ) * 1e-3

    zimp = 0.0
    for imp in range(irm.N_IMPURITIES):
        if irm.impurity_arr_z[imp] > 2:
            zimp += (
                impurity_radiation.zav_of_te(
                    imp, np.array([pv.temp_plasma_electron_vol_avg_kev])
                ).squeeze()
                * (irm.f_nd_impurity_electron_array[imp])
            )
    f_alpha = pv.nd_plasma_alphas_vol_avg / ne
    dne_dt = (fe / vol) / (1 - pv.f_nd_beam_electron - zimp - 2 * f_alpha)

    # Scale back down to nondimensionalised values
    return np.array([dte_dt, dne_dt])


# Parameters to vary
# Original sol at te = 11.26 keV, ne = 8.4e19 m^-3
TE_SOL = 11.26
NE_SOL = 8.4e19
TE_INIT = 10.26
NE_INIT = NE_SOL
P_HCD_PRIMARY_EXTRA_HEAT_MW = 75.0
MOLFLOW_PLASMA_FUELLING_REQUIRED = 7.5e21
FW = 5.0e-6
TRIANG = 0.5

y = np.array([TE_INIT, NE_INIT])
dte_dt, dne_dt = derivatives(
    t=1,
    y=y,
    pinj=P_HCD_PRIMARY_EXTRA_HEAT_MW,
    rfuel=MOLFLOW_PLASMA_FUELLING_REQUIRED,
    fw=FW,
    triang=TRIANG,
)
te_diff = TE_SOL - TE_INIT
t_te = te_diff / dte_dt
print(f"{dte_dt = :.3e} keV s^-1")
print(f"Linearised time to reach te at solution {t_te = :.3e} s")

dte_dt = 2.190e-01 keV s^-1
Linearised time to reach te at solution t_te = 4.567e+00 s


In [43]:
single_run = eval_process(
    te=TE_SOL,
    ne=NE_INIT,
    pinj=P_HCD_PRIMARY_EXTRA_HEAT_MW,
    rfuel=MOLFLOW_PLASMA_FUELLING_REQUIRED,
    fw=FW,
    triang=TRIANG,
)
e_plasma_sol = process.data_structure.physics_variables.e_plasma_beta_thermal
single_run = eval_process(
    te=TE_INIT,
    ne=NE_INIT,
    pinj=P_HCD_PRIMARY_EXTRA_HEAT_MW,
    rfuel=MOLFLOW_PLASMA_FUELLING_REQUIRED,
    fw=FW,
    triang=TRIANG,
)
e_plasma_init = process.data_structure.physics_variables.e_plasma_beta_thermal

ppb = process.constraints.constraint_equation_2().constraint_error * 1e6
vol = process.data_structure.physics_variables.vol_plasma
de_dt = ppb * vol
t_e_plasma = (e_plasma_sol - e_plasma_init) / de_dt
print(f"{de_dt:.3e} W")
print(f"Linearised time to reach solution {t_e_plasma = :.3e}")


2.068e+07 W
Linearised time to reach solution t_e_plasma = 5.168e+00
